In [0]:
CREATE OR REPLACE TABLE support.bugzilla.team_queues AS
SELECT * FROM read_files('/Volumes/support/bugzilla/raw/team_queues.csv', format => 'csv', header => true);

In [0]:
CREATE OR REPLACE TABLE support.bugzilla.silver_bugs AS
SELECT b.id, b.summary, b.description, b.severity, b.status,
       to_timestamp(b.creation_time) AS created_at,
       to_timestamp(b.cf_last_resolved) AS resolved_at,
       t.team,
       b.severity IN ('S1','S2') AS is_urgent
FROM support.bugzilla.bronze_bugs b
JOIN support.bugzilla.team_queues t
  ON b.product = t.product AND b.component = t.component
WHERE b.summary NOT LIKE 'Intermittent%' AND b.summary NOT LIKE 'Perma%'  -- drop robot-filed bugs (hint: their summary starts with 'Intermittent' or 'Perma')

In [0]:
SELECT team, COUNT(*) FROM support.bugzilla.silver_bugs
GROUP BY team
ORDER BY COUNT(*) DESC

In [0]:
SELECT team,
       ROUND(100 * SUM(CASE WHEN is_urgent THEN 1 ELSE 0 END) / COUNT(*), 2) AS urgent_pct
FROM support.bugzilla.silver_bugs
GROUP BY team
ORDER BY urgent_pct DESC

In [0]:
select count(*) from support.bugzilla.silver_bugs